In [0]:
from pyspark.sql.functions import split, col, regexp_replace
from pyspark.sql.functions import to_date, col, when, hour
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import explode

In [0]:
b_trans_df = spark.read.table("`jarvis-catalog`.bronze.transactions_data")
b_cards_df = spark.read.table("`jarvis-catalog`.bronze.cards_data")
b_users_df = spark.read.table("`jarvis-catalog`.bronze.users_data")

TRANSACTIONS DATA

In [0]:
%sql
SELECT * FROM `jarvis-catalog`.landing.transactions_data LIMIT 10;

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01T00:01:00.000Z,1556,2972,-77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475328,2010-01-01T00:02:00.000Z,561,4575,14.5700,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,null
7475329,2010-01-01T00:02:00.000Z,1129,102,80.0000,Swipe Transaction,27092,Vista,CA,92084.0,4829,null
7475331,2010-01-01T00:05:00.000Z,430,2860,200.0000,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,null
7475332,2010-01-01T00:06:00.000Z,848,3915,46.4100,Swipe Transaction,13051,Harwood,MD,20776.0,5813,null
7475333,2010-01-01T00:07:00.000Z,1807,165,4.8100,Swipe Transaction,20519,Bronx,NY,10464.0,5942,null
7475334,2010-01-01T00:09:00.000Z,1556,2972,77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475335,2010-01-01T00:14:00.000Z,1684,2140,26.4600,Online Transaction,39021,ONLINE,null,null,4784,null
7475336,2010-01-01T00:21:00.000Z,335,5131,261.5800,Online Transaction,50292,ONLINE,null,null,7801,null
7475337,2010-01-01T00:21:00.000Z,351,1112,10.7400,Swipe Transaction,3864,Flushing,NY,11355.0,5813,null


Dropping Duplicates, Handling NULLs

In [0]:
# Drop exact duplicate rows
df_cleaned = b_trans_df.dropDuplicates()
df_cleaned = b_trans_df.dropDuplicates(["id"])
# Remove rows where crucial columns are null
df_cleaned = df_cleaned.dropna(subset=["id", "client_id", "card_id"])
# Fill missing values
df_cleaned = df_cleaned.fillna({"merchant_city": "UNKNOWN", "merchant_state": "UNKNOWN",  "zip": 0})
df_cleaned = df_cleaned.withColumn("time_period", when(hour(col("date")) < 12, "AM").otherwise("PM"))
df_cleaned = df_cleaned.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

In [0]:
mcc_json_df = spark.read \
    .option("mode", "PERMISSIVE") \
    .option("multiLine", "true") \
    .json("/Volumes/jarvis-catalog/bronze/json_files/mcc_codes.json")

row = mcc_json_df.first().asDict()
rows = [Row(MCC=col, Description=value) for col, value in row.items()]
mcc_df = spark.createDataFrame(rows)
mcc_df.show(truncate=False)

+----+-----------------------------------------------+
|MCC |Description                                    |
+----+-----------------------------------------------+
|1711|Heating, Plumbing, Air Conditioning Contractors|
|3000|Steelworks                                     |
|3001|Steel Products Manufacturing                   |
|3005|Miscellaneous Metal Fabrication                |
|3006|Miscellaneous Fabricated Metal Products        |
|3007|Coated and Laminated Products                  |
|3008|Steel Drums and Barrels                        |
|3009|Fabricated Structural Metal Products           |
|3058|Tools, Parts, Supplies Manufacturing           |
|3066|Miscellaneous Metals                           |
|3075|Bolt, Nut, Screw, Rivet Manufacturing          |
|3132|Leather Goods                                  |
|3144|Floor Covering Stores                          |
|3174|Upholstery and Drapery Stores                  |
|3256|Brick, Stone, and Related Materials            |
|3260|Pott

In [0]:
schema = StructType([
    StructField(
        "target",
        MapType(StringType(), StringType()),
        True
    )
])

train_fraud_df = (
    spark.read
         .schema(schema)
         .json("/Volumes/jarvis-catalog/bronze/json_files/train_fraud_labels.json")
)


train_fraud_df = train_fraud_df.select(
    explode("target").alias("id", "fraud")
)

In [0]:
df_cleaned = df_cleaned.withColumn("zip", col("zip").cast("int")) \
               .withColumn("amount", F.round(F.col("amount").cast("double"), 2))

In [0]:
df_cleaned = (
    df_cleaned.join(
        mcc_df,
        df_cleaned.mcc == mcc_df.MCC,
        "left"
    )
    .drop(mcc_df.MCC) 
)

df_cleaned = (
    df_cleaned.join(
        train_fraud_df,
        df_cleaned.id == train_fraud_df.id,
        "left"
    )
    .drop(train_fraud_df.id)
)

df_cleaned = df_cleaned.fillna({"fraud": "Unknown"})


cols = df_cleaned.columns

seen = set()
new_cols = []

for col in cols:
    if col == "Description":
        if col not in seen:
            new_cols.append(col)
            seen.add(col)
    else:
        new_cols.append(col)

df_cleaned = df_cleaned.select(*new_cols)
s_trans_df = df_cleaned.withColumnRenamed("Description", "merchant_category")

s_trans_df.show()

+-------+----------+---------+-------+------+------------------+-----------+---------------+--------------+-----+----+------+-----------+--------------------+-----+
|     id|      date|client_id|card_id|amount|          use_chip|merchant_id|  merchant_city|merchant_state|  zip| mcc|errors|time_period|   merchant_category|fraud|
+-------+----------+---------+-------+------+------------------+-----------+---------------+--------------+-----+----+------+-----------+--------------------+-----+
|7483274|2010-01-03|      477|   3419|  3.05| Swipe Transaction|      55060|       Guilford|            ME| 4443|5812|  NULL|         AM|Eating Places and...|   No|
|7477360|2010-01-01|      714|   2616|  6.32| Swipe Transaction|      41260|      West Linn|            OR|97068|5541|  NULL|         PM|    Service Stations|   No|
|7478274|2010-01-01|     1075|   3287| 31.67|Online Transaction|      16798|         ONLINE|       UNKNOWN|    0|4121|  NULL|         PM|Taxicabs and Limo...|   No|
|7475782|2

CARDS DATA

In [0]:
%sql
SELECT * FROM `jarvis-catalog`.landing.cards_data LIMIT 10;

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,true,2,33900.0000,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,true,1,11600.0000,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,true,1,19948.0000,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,true,2,16400.0000,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,true,2,19439.0000,01/1997,2007,No
5,619,Visa,Debit,4657824650820465,04/2024,245,true,2,21883.0000,01/1997,2012,No
6,1046,Amex,Credit,394584924614148,02/1999,302,true,2,9400.0000,01/1998,2011,No
7,511,Mastercard,Debit,5585238056278288,03/2005,749,true,1,9664.0000,01/1998,2011,No
8,1107,Mastercard,Credit,5462760953855576,09/2021,665,false,2,10300.0000,01/1998,2006,No
9,1046,Amex,Credit,357982644067712,09/2020,72,true,1,13000.0000,01/1999,2005,No


In [0]:
# Drop exact duplicate rows
df_cleaned = b_cards_df.dropDuplicates()
df_cleaned = b_cards_df.dropDuplicates(["id"])
# Remove rows where crucial columns are null
df_cleaned = df_cleaned.dropna(subset=["id", "client_id"])
# Fill missing values
df_cleaned = df_cleaned.fillna({"card_brand": "UNKNOWN", "card_type": "UNKNOWN", "card_number": 0, "expires": "00/0000", "cvv": 0, "num_cards_issued": -1, "credit_limit": -1.0, "acct_open_date": "00/0000", "year_pin_last_changed": 0000, "card_on_dark_web": "UNKNOWN"})
from pyspark.sql.functions import split, col

df_cleaned = (
    df_cleaned.withColumn("expire_month", split(col("expires"), "/")[0].cast("int"))
      .withColumn("expire_year", split(col("expires"), "/")[1].cast("int"))
      .withColumn("acct_open_month", split(col("acct_open_date"), "/")[0].cast("int"))
      .withColumn("acct_open_year", split(col("acct_open_date"), "/")[1].cast("int"))
)

s_cards_df = df_cleaned.withColumn("credit_limit", F.round(F.col("credit_limit").cast("double"), 2))

USERS DATA

In [0]:
%sql
SELECT * FROM `jarvis-catalog`.landing.users_data LIMIT 10;

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1


In [0]:
# Drop exact duplicate rows
df_cleaned = b_users_df.dropDuplicates()
df_cleaned = b_users_df.dropDuplicates(["id"])
# Remove rows where crucial columns are null
df_cleaned = df_cleaned.dropna(subset=["id"])
# Fill missing values
df_cleaned = df_cleaned.fillna({"current_age": 0, "retirement_age": 0, "birth_year": 0, "birth_month": 0,"gender": "UNKNOWN", "address": "UNKNOWN"})

s_users_df = df_cleaned.withColumn("per_capita_income", regexp_replace(col("per_capita_income"), "\\$", "").cast("double")).withColumn("yearly_income", regexp_replace(col("yearly_income"), "\\$", "").cast("double")).withColumn("total_debt", regexp_replace(col("total_debt"), "\\$", "").cast("double")).drop("rescued_data")

SAVING THE SILVER TABLES 

In [0]:
spark.sql("DROP TABLE IF EXISTS `jarvis-catalog`.silver.transactions_data")
s_trans_df.write.mode("overwrite").saveAsTable("`jarvis-catalog`.silver.transactions_data")
s_cards_df.write.mode("overwrite").saveAsTable("`jarvis-catalog`.silver.cards_data")
s_users_df.write.mode("overwrite").saveAsTable("`jarvis-catalog`.silver.users_data")